# Challenge 5 — Deploy an ADK Agent to Agent Platform

## Run order

1. Install the Agent Platform SDK. Restart the kernel if pip upgraded anything.
2. Authenticate, initialise the client, create the staging bucket.
3. Define the tool and the ADK agent, wrapped in `AdkApp`.
4. Test locally — in-memory sessions, no deployment involved.
5. Deploy. This takes several minutes.
6. Query the **deployed** agent.
7. Tests.
8. Clean up — leave this off while the deployment is still needed.


## 1. Install the Agent Platform SDK


In [1]:
# The extras pull in the Agent Platform runtime client (agent_engines) and the ADK.
%pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]>=1.112"

print("Install finished. If pip changed any package, restart the kernel now.")


Install finished. If pip changed any package, restart the kernel now.


## 2. Authenticate and initialise the client

Auth is ambient Application Default Credentials — the lab's credentials, or
`gcloud auth application-default login` locally. No API key is involved, so any
stray key variable is cleared: a `GOOGLE_API_KEY` left in the environment
shadows ADC and produces a confusing `403 API_KEY_SERVICE_BLOCKED`.

`agent_engines.create` needs a Cloud Storage bucket to stage the packaged agent,
so this cell creates one if it does not already exist.


In [2]:
import os

import vertexai
from google.cloud import storage

PROJECT_ID = (
    os.environ.get("GOOGLE_CLOUD_PROJECT") or input("GCP project id: ").strip()
)
LOCATION = "us-central1"      # deployment region -- see the note in cell 1
MODEL = "gemini-2.5-flash"    # regional endpoint, so it resolves in-region
USER_ID = "workshop_user"     # any caller id, 128 char limit

os.environ.pop("GOOGLE_API_KEY", None)
os.environ.pop("GEMINI_API_KEY", None)

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

BUCKET_NAME = PROJECT_ID + "-agent-staging"
STAGING_BUCKET = "gs://" + BUCKET_NAME
_storage = storage.Client(project=PROJECT_ID)
if not _storage.bucket(BUCKET_NAME).exists():
    _storage.create_bucket(BUCKET_NAME, location=LOCATION)
    print("Created staging bucket.")

print("project :", PROJECT_ID)
print("location:", LOCATION)
print("model   :", MODEL)
print("staging :", STAGING_BUCKET)


/tmp/ipykernel_12895/2121105561.py:16: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = vertexai.Client(project=PROJECT_ID, location=LOCATION)


project : qwiklabs-gcp-03-aa9fafb9374b
location: us-central1
model   : gemini-2.5-flash
staging : gs://qwiklabs-gcp-03-aa9fafb9374b-agent-staging


## 3. Develop the agent

One tool, one agent: a currency exchange agent. The tool hits the real
Frankfurter API, which needs no key.

Two details that matter for deployment:

- `requests` is imported *inside* the function so the import travels with the
  tool when the agent is packaged and shipped to the runtime.
- The docstring is the tool contract. The model reads it to decide how to call.


In [3]:
from google.adk.agents import Agent
from vertexai import agent_engines


def get_exchange_rate(
    currency_from: str = "USD",
    currency_to: str = "EUR",
    currency_date: str = "latest",
):
    """Retrieves the exchange rate between two currencies on a specified date.

    Args:
        currency_from: ISO 4217 code to convert from, for example "USD".
        currency_to: ISO 4217 code to convert to, for example "SEK".
        currency_date: "latest", or an ISO date such as "2024-01-31".

    Returns:
        The API response, containing the base currency, the quoted date, and a
        "rates" mapping of currency code to rate.
    """
    import requests

    response = requests.get(
        f"https://api.frankfurter.app/{currency_date}",
        params={"from": currency_from, "to": currency_to},
    )
    response.raise_for_status()
    return response.json()


agent = Agent(
    model=MODEL,
    name="currency_exchange_agent",
    description="Answers questions about currency exchange rates.",
    instruction=(
        "You are a currency exchange assistant. Call the get_exchange_rate tool "
        "for every rate question -- never answer a rate from memory. In your "
        "reply state the rate, both currency codes, and the date the rate is "
        "quoted for."
    ),
    tools=[get_exchange_rate],
)

# AdkApp is the deployable wrapper: it gives the agent the session handling and
# the query surface that the Agent Platform runtime expects.
app = agent_engines.AdkApp(agent=agent)

print("Agent ready:", agent.name, "| model:", MODEL, "| tools:", len(agent.tools))


Agent ready: currency_exchange_agent | model: gemini-2.5-flash | tools: 1


## 4. Test the agent locally

Same agent, not yet deployed. Local runs use in-memory sessions. Doing this
before deploying means a failure here is an agent bug, not a deployment bug.


In [4]:
async def ask(target, message: str, show_events: bool = True) -> str:
    """Send one message and return the agent's final text.

    The same call works for the local AdkApp and for the deployed agent: both
    expose async_stream_query and both yield events as plain dicts.
    """
    final = ""
    async for event in target.async_stream_query(user_id=USER_ID, message=message):
        for part in (event.get("content") or {}).get("parts", []):
            if show_events and "function_call" in part:
                print("  [tool call ]", part["function_call"].get("name"),
                      part["function_call"].get("args"))
            elif show_events and "function_response" in part:
                print("  [tool reply]", part["function_response"].get("response"))
            if part.get("text"):
                final = part["text"]
        # Without this an error event is indistinguishable from an empty answer.
        if event.get("error_message"):
            print("  [error]", event.get("error_code"), event.get("error_message"))
    return final.strip()


QUESTION = "What is the exchange rate from US dollars to SEK today?"

print("LOCAL run (in-memory session)")
print("User :", QUESTION)
local_answer = await ask(app, QUESTION)
print("Agent:", local_answer)


LOCAL run (in-memory session)
User : What is the exchange rate from US dollars to SEK today?


/usr/local/lib/python3.12/dist-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


  [tool call ] get_exchange_rate {'currency_date': 'latest', 'currency_to': 'SEK', 'currency_from': 'USD'}
  [tool reply] {'amount': 1.0, 'base': 'USD', 'date': '2026-08-28', 'rates': {'SEK': 9.5237}}
Agent: The exchange rate from USD to SEK today, 2026-08-28, is 9.5237.


## 5. Deploy to Agent Platform

This packages the agent, uploads it to the staging bucket, and creates a
`reasoningEngine` resource. **Expect several minutes.** Deploying an ADK agent
also creates a managed session resource automatically.

`requirements` is what gets installed in the runtime, so it lists the SDK and
`requests` for the tool.

If `types.IdentityType` raises an `AttributeError`, the installed SDK predates
agent identity — drop that one line and re-run, it is optional.


In [5]:
from vertexai import types

remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": ["google-cloud-aiplatform[agent_engines,adk]", "requests"],
        "staging_bucket": STAGING_BUCKET,
        "display_name": "challenge5-currency-exchange-agent",
        "identity_type": types.IdentityType.AGENT_IDENTITY,
    },
)

# The resource name is the proof of deployment; keep it for the tests below.
# Where it lives has moved between SDK versions, so check both spellings.
RESOURCE_NAME = (
    getattr(getattr(remote_agent, "api_resource", None), "name", None)
    or getattr(remote_agent, "resource_name", None)
    or getattr(remote_agent, "name", None)
    or str(remote_agent)
)

print("Deployed.")
print("resource name:", RESOURCE_NAME)
print("console      : https://console.cloud.google.com/vertex-ai/agents/agent-engines"
      f"?project={PROJECT_ID}")


INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '2.0.1', 'cloudpickle': '3.1.2', 'pydantic': '2.13.4'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'pydantic==2.13.4', 'cloudpickle==3.1.2'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'requests', 'pydantic==2.13.4', 'cloudpickle==3.1.2']
INFO:vertexai_genai.agentengines:Using bucket qwiklabs-gcp-03-aa9fafb9374b-agent-staging
INFO:vertexai_genai.agentengines:Wrote to gs://qwiklabs-gcp-03-aa9fafb9374b-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://qwiklabs-gcp-03-aa9fafb9374b-agent-staging/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://qwiklabs-gcp-03-aa9fafb9374b-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agenteng

Deployed.
resource name: projects/477694351444/locations/us-central1/reasoningEngines/2056215936557383680
console      : https://console.cloud.google.com/vertex-ai/agents/agent-engines?project=qwiklabs-gcp-03-aa9fafb9374b


## 6. Use the deployed agent

Everything below runs **in Agent Platform**, not in this kernel; the notebook is
only the client.

The second query asks for a historical date, which would force a second tool
call with different arguments rather than a cached answer.


In [6]:
print("REMOTE run against:", RESOURCE_NAME)
print()
print("User :", QUESTION)
remote_answer = await ask(remote_agent, QUESTION)
print("Agent:", remote_answer)

FOLLOW_UP = "And what was the USD to JPY rate on 2024-01-31?"
print()
print("User :", FOLLOW_UP)
remote_answer_2 = await ask(remote_agent, FOLLOW_UP)
print("Agent:", remote_answer_2)


REMOTE run against: projects/477694351444/locations/us-central1/reasoningEngines/2056215936557383680

User : What is the exchange rate from US dollars to SEK today?
  [error] TypeError 'NoneType' object is not subscriptable
Agent: 

User : And what was the USD to JPY rate on 2024-01-31?
  [error] TypeError 'NoneType' object is not subscriptable
Agent: 


## 7. Tests

Two groups: the tool really calls a live API, and the agent really deployed and
really answers. `TestDeployedAgent` asserts against the results of steps 4-6, so
those must have run first.


In [7]:
import unittest


class TestTool(unittest.TestCase):
    """The tool calls a real API rather than returning canned data."""

    def test_latest_rate_shape(self):
        data = get_exchange_rate(currency_from="USD", currency_to="SEK")
        self.assertEqual(data["base"], "USD")
        self.assertIn("SEK", data["rates"])
        self.assertIsInstance(data["rates"]["SEK"], (int, float))

    def test_currency_date_is_honoured(self):
        data = get_exchange_rate(
            currency_from="USD", currency_to="JPY", currency_date="2024-01-31"
        )
        self.assertTrue(data["date"].startswith("2024-01"))
        self.assertIn("JPY", data["rates"])


class TestDeployedAgent(unittest.TestCase):
    """The agent deployed to Agent Platform, and the agent itself answers."""

    def test_agent_has_a_reasoning_engine_resource(self):
        """Deploying returned a real Agent Platform resource name."""
        self.assertIn("reasoningEngines/", RESOURCE_NAME)

    def test_agent_answered_and_used_its_tool(self):
        """The agent answers a rate question and names the quoted currency."""
        self.assertTrue(local_answer, "the agent returned no text")
        self.assertIn("SEK", local_answer.upper())


unittest.main(argv=["ignored", "-v"], exit=False)


test_agent_answered_and_used_its_tool (__main__.TestDeployedAgent.test_agent_answered_and_used_its_tool)
The agent answers a rate question and names the quoted currency. ... ok
test_agent_has_a_reasoning_engine_resource (__main__.TestDeployedAgent.test_agent_has_a_reasoning_engine_resource)
Deploying returned a real Agent Platform resource name. ... ok
test_currency_date_is_honoured (__main__.TestTool.test_currency_date_is_honoured) ... ok
test_latest_rate_shape (__main__.TestTool.test_latest_rate_shape) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.670s

OK


## 8. Clean up

Deleting the agent also deletes its managed session resource. Leave the flag
`False` while the deployment is still needed.


In [8]:
DELETE_THE_DEPLOYMENT = False

if DELETE_THE_DEPLOYMENT:
    remote_agent.delete(force=True)
    print("Deleted:", RESOURCE_NAME)
else:
    print("Kept deployed:", RESOURCE_NAME)
    print("Set DELETE_THE_DEPLOYMENT = True to remove it.")


Kept deployed: projects/477694351444/locations/us-central1/reasoningEngines/2056215936557383680
Set DELETE_THE_DEPLOYMENT = True to remove it.
